In [4]:
import pandas as pd 
import sqlite3

In [ ]:
## Load the raw data
df=pd.read_csv('superstore.csv', encoding='latin1')

## connecting sqlite
conn=sqlite3.connect("bi_warehouse.db")


## dim date
dim_date=df[['Order Date']].drop_duplicates().reset_index(drop=True)
dim_date.columns=['full_date']
dim_date['full_date']=pd.to_datetime(dim_date['full_date'])
dim_date['date_key']=dim_date.index+1
dim_date['day']=dim_date['full_date'].dt.day
dim_date['month']=dim_date['full_date'].dt.month
dim_date['quarter']=dim_date['full_date'].dt.quarter
dim_date['year']=dim_date['full_date'].dt.year
dim_date.to_sql('dim_date',conn,if_exists='replace',index=False)


## dim customer
dim_customer=df[['Customer ID','Customer Name','Segment']].drop_duplicates().reset_index(drop=True)
dim_customer.columns=['customer_id', 'customer_name', 'segemnt']
dim_customer['customer_key']=dim_customer.index+1
dim_customer.to_sql('dim_customer',conn, if_exists='replace',index=False)

## dim product
dim_product=df[['Product ID', 'Product Name','Category','Sub-Category']].drop_duplicates().reset_index(drop=True)
dim_product.columns=['product_id','product_name','category','sub_category']
dim_product['product_key']=dim_product.index+1
dim_product.to_sql('dim_product', conn, if_exists='replace',index=False)


## dim region
dim_region=df[['City','State','Region','Country']].drop_duplicates().reset_index(drop=True)
dim_region.columns=['city','state','region','country']
dim_region['region_key']=dim_region.index+1
dim_region.to_sql('dim_region',conn, if_exists='replace',index=False)

### Fact_sales
fact=df.merge(dim_date,left_on=pd.to_datetime(df['Order Date']), right_on='full_date')[['Order ID','date_key','Customer ID','Product ID','City','Sales','Profit','Quantity','Discount']]
fact=fact.merge(dim_customer[['customer_id','customer_key']], left_on='Customer ID', right_on='customer_id')
fact=fact.merge(dim_product[['product_id','product_key']],left_on='Product ID',right_on='product_id')
fact=fact.merge(dim_region[['city','region_key']],left_on='City', right_on='city')
fact_sales=fact[['date_key','customer_key','product_key','region_key','Sales','Profit','Quantity','Discount']].reset_index(drop=True)
fact_sales.columns=['date_key','customer_key','product_key','region_key','sales_amount','profit','quantity','discount']
fact_sales['sales_id']=fact_sales.index+1
fact_sales.to_sql('fact_sales',conn, if_exists='replace',index=False)

print("Star Schema created Successfully!")
conn.close()

Star Schema created Successfully!
